# Mosaic Viewer — Doha, Qatar (City-Scale BigTIFFs)

**Case study:** Doha, Qatar (25.2886°N, 51.531°E) — municipality boundary via OSM Nominatim (query `Doha, Qatar`, OSM ID 27332, UTM 39N). Mosaic `run_mosaic.py` grids 4.12 km tiles on 4 km step, center-out, per-tile `run_location` subprocesses, `manifest.json` + `flock`, `gdalbuildvrt` + `gdalwarp` BigTIFFs (`<Place>_<date>_S2SR_*_1m.tif`, `COMPRESS=NONE`, `BIGTIFF=YES`).

**This notebook is Doha-only — Sousse LST is in `03`, Huntington oil in `02`, general demo in `01`.**

Visualizes `outputs/QA/<date>/mosaic-*/` BigTIFFs, downsampled preview, and `manifest.json` (tiles, retries, validation). Uses `folium` for boundary + `rasterio` windowed reads (no full mosaic load).


In [1]:
from pathlib import Path
import glob, json
ROOT = Path.cwd()
if not (ROOT / "scripts" / "run_mosaic.py").exists():
    ROOT = Path.cwd().parent
if not (ROOT / "scripts" / "run_mosaic.py").exists():
    ROOT = Path("/home/khlaifiabilel/NeuralQ/neuralq-s2sr-core")
cands = sorted(glob.glob(str(ROOT / "outputs" / "QA" / "*" / "mosaic-*")))  # Doha only
mosaic = Path(cands[-1]) if cands else None
print("latest mosaic", mosaic)
if mosaic:
    print(list(mosaic.glob("*.tif"))[:3])
    print((mosaic / ".work" / "manifest.json").exists(), (mosaic / "README.md").exists())
else:
    print("No mosaic yet — run: python scripts/run_mosaic.py --date 2026-08-14 --boundary-query 'Lyon, France' --osm-id 35238")


latest mosaic /home/khlaifiabilel/NeuralQ/neuralq-s2sr-core/outputs/QA/2026-08-14/mosaic-20260814-f7e66115
[]
True False


## 1. Manifest — tiles, retries, validation

In [2]:
import json
from pathlib import Path
if mosaic and (mosaic / ".work" / "manifest.json").exists():
    m=json.loads((mosaic / ".work" / "manifest.json").read_text())
    print(f"state {m.get('state')}  tiles {len(m.get('tiles',[]))}  products {list(m.get('products',{}).keys())}")
    import pandas as pd
    df=pd.DataFrame([{"id":t["id"], "status":t["status"], "attempts":t["attempts"], "lon":t["longitude"], "lat":t["latitude"]} for t in m["tiles"][:8]])
    display(df)
elif mosaic:
    print(open(mosaic/"README.md").read().splitlines()[:20])


state planned  tiles 34  products []


,id,status,attempts,lon,lat
0,tile-001,pending,0,51.536267,25.261395
1,tile-002,pending,0,51.496545,25.261534
2,tile-003,pending,0,51.536426,25.297516
3,tile-004,pending,0,51.496692,25.297656
4,tile-005,pending,0,51.536108,25.225273
5,tile-006,pending,0,51.575989,25.261245
6,tile-007,pending,0,51.496398,25.225412
7,tile-008,pending,0,51.576160,25.297367


## 2. Preview — downsampled TCI + boundary
The mosaic preview is `*_preview.tif` (2048 px wide, from TCI). Boundary is `boundary.geojson` in `.work`.


In [3]:
import rasterio, matplotlib.pyplot as plt, json
from pathlib import Path
if mosaic:
    preview = sorted(mosaic.glob("*preview.tif"))
    preview = preview[0] if preview else None
    if preview and preview.exists():
        with rasterio.open(preview) as s:
            w=512; rgb=s.read(window=rasterio.windows.Window(s.width//2-256, s.height//2-256, w, w)).transpose(1,2,0)
            plt.figure(figsize=(6,6)); plt.imshow(rgb); plt.title(f"{preview.name} preview"); plt.axis("off"); plt.show()
        print(preview, s.width, s.height)
    else:
        print("No preview yet")
    # Folium boundary if available
    bpath = mosaic / ".work" / "boundary.geojson" if mosaic else None
    if bpath and bpath.exists():
        import folium
        gj=json.loads(bpath.read_text())
        # crude center from first feature
        coords=gj["features"][0]["geometry"]["coordinates"][0][0] if gj["features"][0]["geometry"]["type"]=="Polygon" else gj["features"][0]["geometry"]["coordinates"][0][0][0]
        lon, lat = coords[0], coords[1]
        m=folium.Map(location=[lat, lon], zoom_start=11)
        folium.GeoJson(gj, name="boundary").add_to(m); m
    else:
        print("No boundary.geojson (mosaic not started)")


No preview yet


## 3. Windowed read — BigTIFF without loading it all
Mosaics are ~tens of GB at 1 m. Use `rasterio.windows.Window` to read a 1024×1024 chip.

In [4]:
import rasterio
from pathlib import Path
if mosaic:
    ms = sorted(mosaic.glob("*S2SR_MS_1m.tif"))
    ms = ms[0] if ms else None
    if ms and ms.exists():
        with rasterio.open(ms) as s:
            print(f"MS {s.width}x{s.height} {s.count} bands {s.dtypes[0]} {s.crs}")
            w=1024; win=rasterio.windows.Window(s.width//2-512, s.height//2-512, w, w)
            chip=s.read(window=win).astype(float)
            print(f"chip {chip.shape} mean DN {chip.mean():.1f}")
            import matplotlib.pyplot as plt, numpy as np
            rgb=np.dstack([chip[2], chip[1], chip[0]])  # B04,B03,B02
            # quick stretch
            def stretch(b):
                lo,hi=np.percentile(b,(2,98))
                return np.clip((b-lo)*(255/(hi-lo+1e-6)),0,255).astype(np.uint8)
            rgb=np.dstack([stretch(chip[2]), stretch(chip[1]), stretch(chip[0])])
            plt.figure(figsize=(5,5)); plt.imshow(rgb); plt.title("MS 1 m chip (B04/B03/B02)"); plt.axis("off"); plt.show()
    else:
        print("No MS BigTIFF yet")


No MS BigTIFF yet
